# Pretrain Dataset Analysis with Embedding Clustering

This notebook analyzes the pretrain_hq.jsonl dataset using text embeddings and clustering to discover content patterns.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import umap
from sentence_transformers import SentenceTransformer
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']  # Support Chinese characters

## 1. Load Pretrain Dataset

In [ ]:
def load_jsonl(file_path):
    """Load JSONL file and return list of records"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data.append(json.loads(line.strip()))
            except json.JSONDecodeError as e:
                print(f"Error parsing line: {e}")
    return data

# Load the pretrain dataset
pretrain_data = load_jsonl('dataset/pretrain_hq.jsonl')
print(f"Loaded {len(pretrain_data)} records from pretrain_hq.jsonl")

# Extract text content
texts = [item['text'] for item in pretrain_data if 'text' in item]
print(f"Extracted {len(texts)} text samples")

# Show sample
print(f"\nSample text (first 300 chars):")
print(texts[0][:300] + "...")

## 2. Basic Text Statistics

In [ ]:
# Calculate text lengths
text_lengths = [len(text) for text in texts]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(text_lengths, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Text Length (characters)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Text Length Distribution', fontsize=14, fontweight='bold')
axes[0].axvline(np.mean(text_lengths), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(text_lengths):.0f}')
axes[0].axvline(np.median(text_lengths), color='green', linestyle='--', linewidth=2, label=f'Median: {np.median(text_lengths):.0f}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(text_lengths, vert=True)
axes[1].set_ylabel('Text Length (characters)', fontsize=12)
axes[1].set_title('Text Length Box Plot', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nText Length Statistics:")
print(f"  Total samples: {len(texts):,}")
print(f"  Mean: {np.mean(text_lengths):.2f}")
print(f"  Median: {np.median(text_lengths):.2f}")
print(f"  Min: {np.min(text_lengths):,}")
print(f"  Max: {np.max(text_lengths):,}")
print(f"  Std: {np.std(text_lengths):.2f}")
print(f"  25th percentile: {np.percentile(text_lengths, 25):.2f}")
print(f"  75th percentile: {np.percentile(text_lengths, 75):.2f}")

## 3. Sample Data for Embedding Analysis

For computational efficiency, we'll sample a subset of the data for embedding analysis.

In [ ]:
# Sample data for embedding analysis (adjust sample_size based on your computational resources)
sample_size = min(5000, len(texts))  # Use up to 5000 samples
np.random.seed(42)
sample_indices = np.random.choice(len(texts), sample_size, replace=False)
sampled_texts = [texts[i] for i in sample_indices]

print(f"Sampled {len(sampled_texts)} texts for embedding analysis")

# Truncate very long texts for embedding (most models have token limits)
max_chars = 512
truncated_texts = [text[:max_chars] for text in sampled_texts]
print(f"Truncated texts to max {max_chars} characters")

## 4. Generate Text Embeddings

Using sentence-transformers to create embeddings. This may take a few minutes depending on sample size.

In [ ]:
# Load a multilingual model that supports Chinese
print("Loading embedding model...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Model loaded successfully!")

# Generate embeddings
print(f"Generating embeddings for {len(truncated_texts)} texts...")
embeddings = model.encode(truncated_texts, show_progress_bar=True, batch_size=32)
print(f"Generated embeddings with shape: {embeddings.shape}")

## 5. Dimensionality Reduction

Reduce embeddings to 2D for visualization using multiple methods.

In [ ]:
# PCA reduction
print("Applying PCA...")
pca = PCA(n_components=2, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")

# t-SNE reduction
print("Applying t-SNE (this may take a while)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeddings_tsne = tsne.fit_transform(embeddings)
print("t-SNE completed!")

# UMAP reduction (with error handling for version compatibility)
embeddings_umap = None
try:
    print("Applying UMAP...")
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    embeddings_umap = reducer.fit_transform(embeddings)
    print("UMAP completed!")
except Exception as e:
    print(f"UMAP failed due to version incompatibility: {e}")
    print("Continuing with PCA and t-SNE only...")

## 6. K-Means Clustering

Apply K-Means clustering to identify content groups.

In [ ]:
# Determine optimal number of clusters using elbow method
inertias = []
silhouette_scores = []
K_range = range(3, 15)

from sklearn.metrics import silhouette_score

print("Finding optimal number of clusters...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(embeddings, kmeans.labels_))

# Plot elbow curve
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method For Optimal k', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score vs Number of Clusters', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Choose optimal k (you can adjust this based on the plots)
optimal_k = 8
print(f"\nUsing k={optimal_k} clusters for final clustering")

In [ ]:
# Apply K-Means with optimal k
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings)

# Cluster statistics
unique_labels, counts = np.unique(cluster_labels, return_counts=True)
print(f"\nCluster Distribution:")
for label, count in zip(unique_labels, counts):
    print(f"  Cluster {label}: {count} samples ({count/len(cluster_labels)*100:.1f}%)")

## 7. Visualize Clusters

Visualize the clusters using different dimensionality reduction techniques.

In [ ]:
# Determine number of subplots based on available methods
n_plots = 3 if embeddings_umap is not None else 2
fig, axes = plt.subplots(1, n_plots, figsize=(7*n_plots, 6))

if n_plots == 2:
    axes = [axes[0], axes[1]]

# PCA visualization
scatter1 = axes[0].scatter(embeddings_pca[:, 0], embeddings_pca[:, 1], 
                          c=cluster_labels, cmap='tab10', alpha=0.6, s=30)
axes[0].set_xlabel('PCA Component 1', fontsize=12)
axes[0].set_ylabel('PCA Component 2', fontsize=12)
axes[0].set_title('Clusters Visualization - PCA', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# t-SNE visualization
scatter2 = axes[1].scatter(embeddings_tsne[:, 0], embeddings_tsne[:, 1], 
                          c=cluster_labels, cmap='tab10', alpha=0.6, s=30)
axes[1].set_xlabel('t-SNE Dimension 1', fontsize=12)
axes[1].set_ylabel('t-SNE Dimension 2', fontsize=12)
axes[1].set_title('Clusters Visualization - t-SNE', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

# UMAP visualization (if available)
if embeddings_umap is not None:
    scatter3 = axes[2].scatter(embeddings_umap[:, 0], embeddings_umap[:, 1], 
                              c=cluster_labels, cmap='tab10', alpha=0.6, s=30)
    axes[2].set_xlabel('UMAP Dimension 1', fontsize=12)
    axes[2].set_ylabel('UMAP Dimension 2', fontsize=12)
    axes[2].set_title('Clusters Visualization - UMAP', fontsize=14, fontweight='bold')
    axes[2].grid(alpha=0.3)
    plt.colorbar(scatter3, ax=axes[2], label='Cluster')

plt.tight_layout()
plt.show()

## 8. Analyze Cluster Content

Extract representative samples from each cluster to understand their characteristics.

In [ ]:
def get_cluster_samples(cluster_id, n_samples=5):
    """Get random samples from a specific cluster"""
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    sample_indices = np.random.choice(cluster_indices, min(n_samples, len(cluster_indices)), replace=False)
    return [sampled_texts[i] for i in sample_indices]

def get_closest_to_centroid(cluster_id, n_samples=3):
    """Get samples closest to cluster centroid"""
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    cluster_embeddings = embeddings[cluster_indices]
    centroid = kmeans.cluster_centers_[cluster_id]
    
    # Calculate distances to centroid
    distances = np.linalg.norm(cluster_embeddings - centroid, axis=1)
    closest_indices = cluster_indices[np.argsort(distances)[:n_samples]]
    
    return [sampled_texts[i] for i in closest_indices]

# Display representative samples from each cluster
for cluster_id in range(optimal_k):
    print(f"\n{'='*100}")
    print(f"CLUSTER {cluster_id} - {counts[cluster_id]} samples ({counts[cluster_id]/len(cluster_labels)*100:.1f}%)")
    print(f"{'='*100}")
    
    # Get samples closest to centroid (most representative)
    representative_samples = get_closest_to_centroid(cluster_id, n_samples=3)
    
    print(f"\n--- Most Representative Samples (closest to centroid) ---")
    for i, sample in enumerate(representative_samples, 1):
        print(f"\nSample {i}:")
        print(sample[:400] + "..." if len(sample) > 400 else sample)
        print()

## 9. Cluster Size Distribution

# Visualize cluster sizes
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Bar chart
axes[0].bar(unique_labels, counts, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Cluster ID', fontsize=12)
axes[0].set_ylabel('Number of Samples', fontsize=12)
axes[0].set_title('Cluster Size Distribution', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3, axis='y')

# Add value labels on bars
for label, count in zip(unique_labels, counts):
    axes[0].text(label, count, f'{count}\n({count/len(cluster_labels)*100:.1f}%)', 
                ha='center', va='bottom', fontsize=10)

# Pie chart
axes[1].pie(counts, labels=[f'Cluster {i}' for i in unique_labels], autopct='%1.1f%%',
           startangle=90, colors=plt.cm.tab10.colors)
axes[1].set_title('Cluster Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Text Length Analysis by Cluster

In [ ]:
# Analyze text lengths by cluster
cluster_lengths = {i: [] for i in range(optimal_k)}
for idx, label in enumerate(cluster_labels):
    cluster_lengths[label].append(len(sampled_texts[idx]))

# Box plot of text lengths by cluster
fig, ax = plt.subplots(figsize=(14, 6))
box_data = [cluster_lengths[i] for i in range(optimal_k)]
bp = ax.boxplot(box_data, labels=[f'C{i}' for i in range(optimal_k)], patch_artist=True)

# Color the boxes
colors = plt.cm.tab10.colors
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_xlabel('Cluster ID', fontsize=12)
ax.set_ylabel('Text Length (characters)', fontsize=12)
ax.set_title('Text Length Distribution by Cluster', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print statistics
print("\nText Length Statistics by Cluster:")
for i in range(optimal_k):
    lengths = cluster_lengths[i]
    print(f"\nCluster {i}:")
    print(f"  Mean: {np.mean(lengths):.2f}")
    print(f"  Median: {np.median(lengths):.2f}")
    print(f"  Min: {np.min(lengths)}")
    print(f"  Max: {np.max(lengths)}")

## 11. Export Clustering Results

In [ ]:
# Create summary statistics
summary_stats = {
    'Total Samples': len(texts),
    'Sampled for Analysis': len(sampled_texts),
    'Number of Clusters': optimal_k,
    'Embedding Dimensions': embeddings.shape[1],
    'Average Text Length': f"{np.mean(text_lengths):.2f} chars",
    'Median Text Length': f"{np.median(text_lengths):.2f} chars"
}

print("\n" + "="*60)
print("ANALYSIS SUMMARY")
print("="*60)
for key, value in summary_stats.items():
    print(f"{key:.<40} {value}")
print("="*60)

In [ ]:
# Create a DataFrame with clustering results
results_dict = {
    'text': sampled_texts,
    'cluster': cluster_labels,
    'text_length': [len(text) for text in sampled_texts],
    'pca_x': embeddings_pca[:, 0],
    'pca_y': embeddings_pca[:, 1],
    'tsne_x': embeddings_tsne[:, 0],
    'tsne_y': embeddings_tsne[:, 1]
}

# Add UMAP columns if available
if embeddings_umap is not None:
    results_dict['umap_x'] = embeddings_umap[:, 0]
    results_dict['umap_y'] = embeddings_umap[:, 1]

results_df = pd.DataFrame(results_dict)

# Save to CSV
results_df.to_csv('pretrain_clustering_results.csv', index=False)
print("Clustering results saved to pretrain_clustering_results.csv")

# Display summary
print(f"\nDataFrame shape: {results_df.shape}")
print(f"\nCluster distribution:")
print(results_df['cluster'].value_counts().sort_index())
print(f"\nFirst few rows:")
print(results_df.head())